# Task 9: Simple FlashAttention (Scalar)

Bot task id: `09_flashattention_scalar`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/09_flashattention_scalar.ipynb)

## Условие

Простая версия FlashAttention. Даны тензоры `q`, `k`, `v` формы `(T,)`. Нужно вычислить

$$s_j = q \cdot k[j], \qquad \text{soft} = \operatorname{softmax}(s), \qquad res = \sum_j \text{soft}[j] \cdot v[j]$$

**Требования**
- обрабатывать последовательность тайлами `B1 < T` в одном ядре (один `pid0` = одна голова);
- стабильный softmax с *running max / renormalization*:
```
m_new = max(m, s_j)
l   = l   * 2**(m - m_new) + 2**(s_j - m_new)
acc = acc * 2**(m - m_new) + v_j * 2**(s_j - m_new)
m = m_new
```
- результат = `acc / l`.

**Как здесь понято условие.** Формулировка оставляет форму выхода открытой. Проверка читает её как в [Triton-Puzzles](https://github.com/srush/Triton-Puzzles): каждый `q[i]` — отдельный запрос, `out[i] = sum_j softmax_j(q[i] * k[j]) * v[j]`, выход формы `(T,)`, `pid0` перебирает `i`. Если бот ждёт другое, поправьте проверку ниже.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `09_flashattention_scalar.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 09_flashattention_scalar.py
import torch
import triton
import triton.language as tl


@triton.jit
def _flashatt_kernel(q_ptr, k_ptr, v_ptr, out_ptr, T, B1: tl.constexpr, LOG2E: tl.constexpr):
    # YOUR CODE HERE
    pass


def flashatt_triton(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "09_flashattention_scalar.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

q, k, v = (torch.randn(200, device="cuda") for _ in range(3))
ref = (torch.softmax(q[:, None] * k[None, :], 1) * v[None, :]).sum(1)
torch.testing.assert_close(sol.flashatt_triton(q, k, v), ref, atol=1e-4, rtol=1e-4)

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("09_flashattention_scalar.py")